# 06 | Relação entre regras e homicídios

Usamos uma regressão linear para examinar se **registros RAND vigentes no fim de um ano** estão associados à taxa de homicídios do ano seguinte. O modelo considera desemprego, renda, diferenças fixas entre estados e diferenças gerais entre anos. Isso ajuda a evitar comparações simples entre estados muito diferentes, mas **não prova impacto direto das leis**. Não existe aqui uma variável “legalização: sim ou não”: a RAND reúne regras diferentes, classificadas como restritivas ou permissivas.

## Bloco 1: Abrir e conferir o painel

Execute antes 01 → 02 → 03 → 04 → 05. O painel deve conter 300 estado-anos.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import importlib
import src.association_model as association_model
importlib.reload(association_model)
fit_within_state = association_model.fit_within_state
painel = pd.read_csv(ROOT / "data/processed/us_gun_policy_panel_2019_2024.csv")
assert painel.shape == (300, 15)
print("Estados:", painel["state"].nunique(), "| Anos:", sorted(painel["year"].unique()))
display(painel[["state", "year", "homicide_rate", "restrictive_records", "permissive_records", "unemployment_rate", "income_per_capita"]].head())

## Bloco 2: Ajustar o modelo de associação

Os coeficientes mostram a diferença média na taxa de homicídios associada a **um registro adicional no fim do ano anterior** de cada tipo, mantendo constantes as outras colunas do modelo. Usamos 250 observações de 2020 a 2024; desemprego e renda também são do ano anterior. O intervalo de 95% representa a incerteza estatística estimada, agrupada por estado. Se o intervalo cruza zero, os dados não indicam uma direção clara. Mesmo se não cruzar, isso não estabelece causalidade.

In [ ]:
resultado = fit_within_state(painel)
display(resultado.round(3))
for termo in ["restrictive_previous_year", "permissive_previous_year"]:
    linha = resultado.set_index("term").loc[termo]
    conclusao = "sem direção clara" if linha["ci_low"] <= 0 <= linha["ci_high"] else "associação com direção estimada"
    print(termo, ":", conclusao)

## Bloco 3: Conferir mudanças e salvar

Mostramos quantos estados tiveram alguma alteração na contagem entre 2019 e 2024. Isso é uma checagem de variação disponível para o modelo, não uma avaliação causal de cada mudança.

In [ ]:
variacao = painel.sort_values(["state", "year"]).copy()
for coluna in ["restrictive_records", "permissive_records"]:
    variacao["mudanca_" + coluna] = variacao.groupby("state")[coluna].diff()
print("Estados com alteração em registros restritivos:", variacao.loc[variacao["mudanca_restrictive_records"].fillna(0).ne(0), "state"].nunique())
print("Estados com alteração em registros permissivos:", variacao.loc[variacao["mudanca_permissive_records"].fillna(0).ne(0), "state"].nunique())
saida = ROOT / "reports/results"
saida.mkdir(parents=True, exist_ok=True)
resultado.to_csv(saida / "association_model.csv", index=False)
print("Resultado salvo em reports/results/association_model.csv")